<a href="https://colab.research.google.com/github/daddmi/used-car-price-model/blob/main/wk2_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Generate Synthetic Used Car Dataset and Save to Google Drive

import numpy as np
import pandas as pd
from datetime import datetime
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Reproducibility
np.random.seed(42)

# Dataset size
n = 5000
current_year = datetime.now().year

# Makes and Models
makes_models = {
    'Toyota': ['Corolla', 'Camry', 'RAV4', 'Prius'],
    'Honda': ['Civic', 'Accord', 'CR-V', 'Fit'],
    'Ford': ['Focus', 'Fiesta', 'Escape', 'Fusion'],
    'BMW': ['3 Series', '5 Series', 'X3', 'X5'],
    'Mercedes': ['C Class', 'E Class', 'GLA', 'GLE'],
    'Hyundai': ['i30', 'Elantra', 'Tucson', 'Santa Fe'],
    'Kia': ['Rio', 'Cerato', 'Sportage', 'Sorento'],
    'Nissan': ['Sentra', 'Altima', 'Qashqai', 'Leaf'],
    'Volkswagen': ['Golf', 'Passat', 'Tiguan', 'Polo'],
    'Audi': ['A3', 'A4', 'Q3', 'Q5']
}

fuel_types = ['petrol', 'diesel', 'hybrid', 'electric']
transmissions = ['manual', 'auto']
conditions = ['excellent', 'good', 'fair', 'poor']

base_price_map = {
    'Toyota': 20000,
    'Honda': 21000,
    'Ford': 15000,
    'BMW': 35000,
    'Mercedes': 38000,
    'Hyundai': 14000,
    'Kia': 12000,
    'Nissan': 13000,
    'Volkswagen': 18000,
    'Audi': 33000
}

model_adj = {
    'Corolla':0.95,'Camry':1.05,'RAV4':1.15,'Prius':1.1,
    'Civic':0.95,'Accord':1.05,'CR-V':1.12,'Fit':0.8,
    'Focus':0.85,'Fiesta':0.7,'Escape':1.0,'Fusion':0.9,
    '3 Series':1.4,'5 Series':1.8,'X3':1.5,'X5':2.2,
    'C Class':1.6,'E Class':2.0,'GLA':1.7,'GLE':2.3,
    'i30':0.9,'Elantra':0.95,'Tucson':1.0,'Santa Fe':1.1,
    'Rio':0.7,'Cerato':0.85,'Sportage':1.0,'Sorento':1.2,
    'Sentra':0.85,'Altima':0.95,'Qashqai':1.0,'Leaf':1.1,
    'Golf':0.95,'Passat':1.05,'Tiguan':1.1,'Polo':0.8,
    'A3':1.5,'A4':1.7,'Q3':1.6,'Q5':2.0
}

rows = []

for i in range(n):

    make = np.random.choice(
        list(makes_models.keys()),
        p=[0.18,0.16,0.12,0.06,0.05,0.12,0.06,0.08,0.09,0.08]
    )

    model = np.random.choice(makes_models[make])

    year = np.random.choice(
        range(current_year - 15, current_year + 1),
        p=np.linspace(1, 0.2, 16) / np.linspace(1, 0.2, 16).sum()
    )

    age = current_year - year

    # Mileage
    base_mileage = np.clip(
        np.random.normal(15000, 5000) * age,
        0,
        400000
    )

    mileage = max(
        0,
        int(base_mileage + np.random.normal(0, 3000))
    )

    # Engine Size
    engine_size = round(
        np.random.choice(
            [1.0,1.2,1.4,1.6,1.8,2.0,2.5,3.0],
            p=[0.08,0.06,0.12,0.20,0.18,0.18,0.12,0.06]
        ),
        1
    )

    fuel = np.random.choice(
        fuel_types,
        p=[0.60,0.20,0.15,0.05]
    )

    transmission = np.random.choice(
        transmissions,
        p=[0.35,0.65]
    )

    previous_owners = np.random.choice(
        [0,1,2,3],
        p=[0.25,0.50,0.20,0.05]
    )

    service_history = np.random.choice(
        ['full','partial','none'],
        p=[0.45,0.40,0.15]
    )

    location = np.random.choice(
        ['Lagos','Abuja','Kano','Port Harcourt',
         'Ibadan','Kaduna','Enugu','Benin'],
        p=[0.20,0.15,0.12,0.12,0.12,0.08,0.11,0.10]
    )

    condition = np.random.choice(
        conditions,
        p=[0.15,0.50,0.25,0.10]
    )

    # Price Calculation
    base = base_price_map[make] * (
        1 + np.random.normal(0, 0.05)
    )

    base *= model_adj.get(model, 1.0)

    price = (
        base * max(0.25, 1 - 0.08 * age)
        - (mileage / 1000) * 50
    )

    condition_multiplier = {
        'excellent':1.05,
        'good':1.00,
        'fair':0.85,
        'poor':0.60
    }

    price *= condition_multiplier[condition]

    if service_history == 'full':
        price *= 1.03
    elif service_history == 'none':
        price *= 0.95

    price *= (1 - 0.03 * previous_owners)

    location_adjustment = {
        'Lagos':1.05,
        'Abuja':1.03,
        'Kano':0.95,
        'Port Harcourt':1.02,
        'Ibadan':0.98,
        'Kaduna':0.96,
        'Enugu':0.97,
        'Benin':0.96
    }

    price *= location_adjustment[location]

    price = round(
        max(800, price + np.random.normal(0, 1000)),
        2
    )

    # Missing Values
    if np.random.rand() < 0.02:
        engine_size = np.nan

    if np.random.rand() < 0.03:
        fuel = np.nan

    if np.random.rand() < 0.01:
        mileage = np.nan

    rows.append({
        'car_id': f'CAR{i+1:05d}',
        'price': price,
        'make': make,
        'model': model,
        'year': year,
        'age': age,
        'mileage': mileage,
        'engine_size': engine_size,
        'fuel_type': fuel,
        'transmission': transmission,
        'condition': condition,
        'previous_owners': previous_owners,
        'service_history': service_history,
        'location': location
    })

# Create DataFrame
df = pd.DataFrame(rows)

# Sort Dataset
df = df.sort_values(
    by=['make', 'year'],
    ascending=[True, False]
).reset_index(drop=True)

# Dataset Information
print("\n===== DATASET SHAPE =====")
print(df.shape)

print("\n===== FIRST 10 ROWS =====")
print(df.head(10))

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

print("\n===== DESCRIPTIVE STATISTICS =====")
print(df.describe(include='all').T)

# Save Files
local_path = '/content/cleaned_cars.csv'
drive_path = '/content/drive/MyDrive/cleaned_cars.csv'

df.to_csv(local_path, index=False)
df.to_csv(drive_path, index=False)

print("\n===================================")
print("Dataset Created Successfully!")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Saved To: {drive_path}")
print("===================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== DATASET SHAPE =====
(5000, 14)

===== FIRST 10 ROWS =====
     car_id     price  make model  year  age  mileage  engine_size fuel_type  \
0  CAR00062  55105.60  Audi    Q3  2026    0   4863.0          1.8    petrol   
1  CAR00317  50071.26  Audi    A3  2026    0      0.0          1.0    petrol   
2  CAR01506  53077.25  Audi    Q3  2026    0      0.0          1.2    petrol   
3  CAR02095  42533.10  Audi    Q3  2026    0   3402.0          1.6    hybrid   
4  CAR02446  40059.20  Audi    A4  2026    0   3405.0          2.0    petrol   
5  CAR03025  56356.09  Audi    A3  2026    0   4337.0          3.0    petrol   
6  CAR04525  38904.84  Audi    Q5  2026    0   5389.0          NaN    petrol   
7  CAR04704  36622.43  Audi    Q5  2026    0   3170.0          1.0    petrol   
8  CAR04811  28356.08  Audi    A3  2026    0      0.0          1.2    hybrid   
9  CAR